# Data preparation for building Uni-Mol-MeCAPs

# Case study: 
## Baldi elec and nuc


In [1]:
# Now unzip necessary data 'QMdata4ML' to current directory
!tar --extract \
    --file=data.tar.xz \
    --xz \
    --strip-components=1 \
    --directory=. \
    data/benchmark

tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.FinderInfo'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemTextContentLanguage'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseVersion'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseLabels'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseConfidences'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemTextContentLanguage'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseVersion'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseLabels'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDIt

In [6]:
import pandas as pd

df_nuc  = pd.read_csv('./benchmark/nucleophilicity.csv', index_col=0)
df_elec = pd.read_csv('./benchmark/electrophilicity.csv', index_col=0)

com_col = ['smiles','atomIdx']
df_nuc  = df_nuc.rename(columns={'MCA r2SCAN-3c SMD(DMSO)//GFN1-xTB ALPB(DMSO)': 'MCA_values'})[com_col+['MCA_values']]
df_elec = df_elec.rename(columns={'MAA r2SCAN-3c SMD(DMSO)//GFN1-xTB ALPB(DMSO)': 'MAA_values'})[com_col+['MAA_values']]

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

for df in (df_nuc, df_elec,):
    base_name = 'nucleophilicity' if 'MCA_values' in df.columns else 'electrophilicity'
    df['split'] = None
    raw_indices = list(range(len(df)))
    
    train_idx, val_idx = train_test_split(raw_indices, test_size=0.1, random_state=0)
    df.loc[df.index[train_idx],'split'] = 'train'
    df.loc[df.index[val_idx],'split'] = 'val'
    df.to_csv(f'./benchmark/{base_name}_whole.csv')

    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    for i, (tr_vl_idx, test_idx) in enumerate(kf.split(raw_indices)):
        df['split'] = None
        train_idx, val_idx = train_test_split(tr_vl_idx, test_size=0.1, random_state=0)
        df.loc[df.index[train_idx],'split'] = 'train'
        df.loc[df.index[val_idx],'split'] = 'val'
        df.loc[df.index[test_idx],'split'] = 'test'
        df.to_csv(f'./benchmark/{base_name}_fold{i}.csv')